# Tutorial 01 (Bonds to Devices) — Why hydrogen comes in pairs

Hydrogen gas is made of H₂ molecules, not single H atoms. This notebook shows why, using the energy of two hydrogen atoms as a function of the distance between them. The curve is **simulated** with a Morse model whose three inputs are **measured** spectroscopic constants for H₂ taken from the literature. No new measurements are introduced.

This is Episode 1 of the *Bonds to Devices* series. The short video for the same episode is made by `make_reel_ep01.py` in this folder, from the same `h2_morse.py` module, so every number in the video can be traced to a cell below.

## Learning objectives

By the end of the notebook, you should be able to:

- read an energy-versus-distance curve and find the bond length and bond energy on it;
- explain why the energy needed to break a bond (D₀) is smaller than the depth of the curve (Dₑ);
- convert between eV per molecule, kJ/mol and cm⁻¹ without losing a factor;
- compare a bond energy with thermal energy at room temperature; and
- say which parts of a simple model agree with experiment and which do not.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

import h2_morse as h2

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")

## 1. Data source and units

The model needs only three measured numbers: the bond length rₑ, the vibrational constant ωₑ (which sets how stiff the bond is) and the well depth Dₑ. A fourth, the measured dissociation energy D₀, is kept aside to test the model. Distances are in picometres (1 pm = 10⁻¹² m) and energies in electronvolts. Spectroscopists quote energies in wavenumbers (cm⁻¹); `h2.EV_PER_CM1` converts them.

In [ ]:
constants = pd.DataFrame({
    "quantity": ["bond length r_e", "vibrational constant omega_e", "anharmonicity omega_e x_e",
                 "well depth D_e", "dissociation energy D_0 (test value)"],
    "value": [h2.BOND_LENGTH_PM, h2.OMEGA_E_CM1, h2.OMEGA_E_XE_CM1,
              h2.WELL_DEPTH_EV, h2.D0_MEASURED_CM1],
    "unit": ["pm", "cm^-1", "cm^-1", "eV", "cm^-1"],
    "used for": ["model input", "model input", "check only", "model input", "check only"],
    "source": ["Huber & Herzberg (1979)", "Huber & Herzberg (1979)", "Huber & Herzberg (1979)",
               "Kolos & Wolniewicz (1968)", "Liu et al. (2009)"],
})
constants

## 2. The energy curve

The Morse curve is $E(r) = D_e\left[\left(1 - e^{-a(r - r_e)}\right)^2 - 1\right]$. Zero energy means two separate atoms at rest, so a negative value means the pair is bound. The width parameter $a$ is not fitted: it is fixed by requiring the curve to have the measured stiffness of the bond.

In [ ]:
distance_pm = np.linspace(40, 400, 3601)
energy_eV = h2.morse_energy_eV(distance_pm)

i_min = np.argmin(energy_eV)
print(f"Width parameter a: {h2.morse_width_per_pm():.5f} per pm")
print(f"Lowest point of the curve: {energy_eV[i_min]:.4f} eV at {distance_pm[i_min]:.1f} pm")
print(f"Energy at 250 pm: {h2.morse_energy_eV(250):.3f} eV")

In [ ]:
levels_eV = h2.morse_level_eV(np.arange(h2.morse_level_count()))

fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.plot(distance_pm, energy_eV, label="Morse curve")
for k, level in enumerate(levels_eV):
    inner, outer = h2.turning_points_pm(level)
    ax.hlines(level, inner, outer, color="tab:orange", lw=0.8,
              label="allowed vibration levels" if k == 0 else None)
ax.axhline(0, color="grey", ls="--", lw=0.8, label="two free atoms")
ax.set_xlabel("Distance between the nuclei (pm)")
ax.set_ylabel("Energy (eV)")
ax.set_xlim(40, 400)
ax.set_ylim(-5.0, 0.8)
ax.set_title("Hydrogen molecule: energy against distance")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

## 3. The bond never sits still

Quantum mechanics does not allow the pair to rest at the bottom of the curve. The lowest allowed state, v = 0, sits above the bottom by the zero-point energy. The molecule swings between two turning points at about 1.3 × 10¹⁴ times per second. The video slows this down so that one swing takes just over a second.

In [ ]:
frequency_Hz = h2.vibration_frequency_Hz()
level0_eV = levels_eV[0]
zero_point_eV = level0_eV + h2.WELL_DEPTH_EV
inner_pm, outer_pm = h2.turning_points_pm(level0_eV)

print(f"Vibration frequency: {frequency_Hz:.3e} Hz")
print(f"Zero-point energy: {zero_point_eV:.3f} eV")
print(f"Lowest level sits at {level0_eV:.3f} eV")
print(f"Classical swing in the lowest level: {inner_pm:.1f} pm to {outer_pm:.1f} pm")
print(f"Bound levels in the Morse model: {h2.morse_level_count()} (measured for H2: 15)")

## 4. How much energy breaks the bond?

Starting from the lowest allowed level, the energy needed to separate the atoms is $D_0 = D_e - E_\text{zero point}$. This is the number a chemist means by the bond energy. It is compared here with the measured value, which the model was not given. The conversion to kJ/mol multiplies by Avogadro's number.

In [ ]:
d0_model_eV = -level0_eV
d0_measured_eV = h2.D0_MEASURED_EV
d0_measured_kJ_per_mol = d0_measured_eV * h2.KJ_PER_MOL_PER_EV
relative_error = (d0_model_eV - d0_measured_eV) / d0_measured_eV

print(f"D0 from the Morse model: {d0_model_eV:.4f} eV")
print(f"D0 measured:             {d0_measured_eV:.4f} eV = {d0_measured_kJ_per_mol:.1f} kJ/mol")
print(f"Difference: {100 * relative_error:+.3f} %")

## 5. Why room temperature cannot pull H₂ apart

The typical thermal energy per molecule is about $k_B T$. The Boltzmann factor $e^{-D_0/k_B T}$ gives a rough idea of how rarely a collision would carry enough energy to break the bond. It is an order-of-magnitude guide, not a reaction rate.

In [ ]:
temperatures_K = np.array([298.15, 1000.0, 3000.0])
thermal_eV = h2.K_B_EV_PER_K * temperatures_K
ratio = d0_measured_eV / thermal_eV
log10_boltzmann = -ratio / np.log(10)

pd.DataFrame({
    "temperature_K": temperatures_K,
    "thermal_energy_eV": thermal_eV.round(4),
    "D0 / kT": ratio.round(1),
    "log10 of Boltzmann factor": log10_boltzmann.round(1),
})

## 6. A picture of the shared electrons

The simplest textbook picture of the shared electrons is the bonding orbital: the sum of two hydrogen 1s orbitals. As the atoms approach, density builds up between the nuclei. This picture is only qualitative. On its own it predicts a bond about 2.7 eV deep and roughly 85 pm long, far from the measured values, which is why the energies in this notebook come from the Morse curve instead.

In [ ]:
x_pm, y_pm = np.meshgrid(np.linspace(-250, 250, 301), np.linspace(-150, 150, 181))
reference = h2.bonding_density(0.0, 0.0, h2.BOND_LENGTH_PM)

fig, axes = plt.subplots(1, 3, figsize=(9.6, 2.6), sharey=True)
for ax, d_pm in zip(axes, [250, 130, h2.BOND_LENGTH_PM]):
    density = h2.bonding_density(x_pm, y_pm, d_pm) / reference
    ax.imshow(np.clip(density, 0, 1), extent=[-250, 250, -150, 150], origin="lower",
              cmap="viridis", vmin=0, vmax=1)
    ax.set_title(f"{d_pm:.0f} pm apart")
    ax.set_xlabel("x (pm)")
axes[0].set_ylabel("y (pm)")
fig.tight_layout()
plt.show()

## 7. Validation checks

Each check compares the model with a number it was not built from, or with a value that must hold by construction.

In [ ]:
# The minimum of the curve must sit at the input bond length, at depth -De.
assert abs(distance_pm[i_min] - h2.BOND_LENGTH_PM) < 0.2
assert np.isclose(energy_eV[i_min], -h2.WELL_DEPTH_EV, atol=1e-4)

# Far apart, the energy must approach zero (two free atoms).
assert abs(h2.morse_energy_eV(2000.0)) < 1e-12

# The model's D0 must match the independent measurement to better than 0.1 %.
assert abs(relative_error) < 1e-3

# Unit check: 1 eV per molecule is about 96.5 kJ/mol, so D0 should be about 432 kJ/mol.
assert 431 < d0_measured_kJ_per_mol < 433

# The Morse anharmonicity is a prediction; it should be within 5 % of the measured 121.34 cm^-1.
quantum_cm1 = h2.OMEGA_E_CM1
morse_anharmonicity_cm1 = quantum_cm1**2 / (4 * h2.WELL_DEPTH_EV / h2.EV_PER_CM1)
print(f"Morse anharmonicity: {morse_anharmonicity_cm1:.1f} cm^-1 (measured {h2.OMEGA_E_XE_CM1})")
assert abs(morse_anharmonicity_cm1 / h2.OMEGA_E_XE_CM1 - 1) < 0.05

print("All checks passed.")

## Interpretation and checks

- **Calculation.** With three measured inputs, the Morse curve puts the lowest energy at 74.1 pm and −4.75 eV, predicts a bond energy D₀ of 4.479 eV, and lands within 0.03 % of the measured 4.478 eV (432 kJ/mol).
- **Interpretation.** At room temperature the thermal energy is about 0.026 eV, roughly 170 times smaller than D₀. This is consistent with hydrogen gas staying as H₂ under everyday conditions. Splitting it takes very high temperatures, light of short wavelength, or a catalyst surface.
- **What the model does not get right.** The Morse curve predicts 17 bound vibration levels where H₂ has 15, and its anharmonicity is about 4 % too large. It also decays too slowly at large distances, so the energy it gives beyond about 200 pm is too negative. The 1s + 1s cloud is a qualitative picture and should not be used for numbers.
- **What this does not show.** A good match for D₀ does not prove the Morse shape is correct everywhere. It shows that a curve with the right depth, position and stiffness near the bottom describes the low-lying levels well.
- **A single bond length?** 74 pm is the bottom of the curve. Because the molecule always vibrates, the bond length you get from its rotational spectrum is slightly longer, about 75 pm.

## Exercise

1. Change the temperature table to include the surface of the Sun (about 5800 K). Is D₀/kT still large?
2. Deuterium (D₂) has the same curve but twice the atomic mass. Change `h2.REDUCED_MASS_AMU` and `h2.OMEGA_E_CM1` (ωₑ scales as 1/√μ) and predict D₀ for D₂. The measured value is 4.556 eV. Explain why D₂ is harder to break.
3. Plot the classical swing of the lowest level against the full curve and mark the two turning points.
4. Open-ended: the video says the pair "never sits still". Write two sentences for a non-scientist that explain zero-point energy without using the word "quantum".

## References

- Huber, K. P., & Herzberg, G. (1979). *Molecular Spectra and Molecular Structure IV: Constants of Diatomic Molecules*. Van Nostrand Reinhold. Values for H₂ as listed in the NIST Chemistry WebBook, SRD 69.
- Kołos, W., & Wolniewicz, L. (1968). Improved theoretical ground-state energy of the hydrogen molecule. *Journal of Chemical Physics*, 49, 404–410.
- Liu, J., Salumbides, E. J., Hollenstein, U., Koelemeij, J. C. J., Eikema, K. S. E., Ubachs, W., & Merkt, F. (2009). Determination of the ionization and dissociation energies of the hydrogen molecule. *Journal of Chemical Physics*, 130, 174306.
- Morse, P. M. (1929). Diatomic molecules according to the wave mechanics. II. Vibrational levels. *Physical Review*, 34, 57–64.
- Tiesinga, E., Mohr, P. J., Newell, D. B., & Taylor, B. N. (2021). CODATA recommended values of the fundamental physical constants: 2018. *Reviews of Modern Physics*, 93, 025010.
- Atkins, P., & de Paula, J. (2014). *Atkins' Physical Chemistry* (10th ed.). Oxford University Press. Chapter on molecular structure (LCAO description of H₂⁺ and H₂).